# 04 — Benchmark causal de los 12 detectores (Fase D)

Este notebook abre la **Fase D**: toma los paneles causales producidos por
`03_preprocesado.ipynb` y reevalúa los doce detectores de la Capa 1 bajo un
protocolo común. La Capa 1 no se modifica; sus implementaciones se reutilizan a
través de `src/detectors/registry.py` y la ejecución vive en `src/benchmark.py`.

> **Objetivo de este notebook.** Ejecutar y guardar evidencia comparable. Aquí no
> se elige todavía un ganador ni se diseña el híbrido final; esa lectura se hace en
> `05_comparacion_detectores.ipynb`.

**Entradas:** `data/processed/pista{A,B}_{diaria,mensual}.parquet`,
`data/benchmark_spec.yaml` y los detectores congelados de `capa1_exploracion/`.

**Salidas:** una fila de métricas y un panel OOS por combinación pista–detector,
además de `manifest.json` y `run_status.csv` en `results/benchmark_v2/`.


In [1]:
import sys, warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')
ROOT = Path.cwd()
while not (ROOT / 'data' / 'benchmark_spec.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import benchmark as bm
from src.detectors import specs_table

pd.set_option('display.max_colwidth', 90)
print('ROOT:', ROOT)
print('Paneles procesados disponibles:', bm.processed_available())


ROOT: C:\Users\Fernando Dapena T\Desktop\TFM\PRUEBAS_DETECCION_REGIMENES_DE_MERCADO_TFM
Paneles procesados disponibles: True


## 1. Qué se va a probar

Se ejecutan los doce detectores oficiales D1–D12, no la extensión HSMM D13. Cada
modelo conserva su familia y su lógica de Capa 1, pero recibe una configuración v2
declarada antes de observar resultados:

- los modelos multivariantes usan un núcleo fijo por pista, evitando entregar todas las
  columnas del panel (35 en A, 77 en B) a modelos de covarianza completa;
- los modelos de varianza y cambio usan el retorno logarítmico natural `SP500_ret`, derivado causalmente de la espina cruda;
- D1 usa VIX en B y volatilidad realizada en A, porque VIX no pertenece a la
  ventana histórica de A; esta variante queda marcada en todos los resultados;
- D2 usa equivalentes v2 de volatilidad, crédito, pendiente y drawdown;
- los detectores caros reducen la frecuencia de reestimación, pero continúan
  produciendo una etiqueta diaria causal dentro de cada bloque.

La tabla siguiente es el contrato efectivo. Si se cambia una fila, debe justificarse
antes de ejecutar y quedar registrada en el manifiesto.


In [2]:
tabla_A = specs_table('A').assign(pista='A')
tabla_B = specs_table('B').assign(pista='B')
display(pd.concat([tabla_A, tabla_B], ignore_index=True)[
    ['pista', 'id', 'detector', 'familia', 'n_features', 'step',
     'ventana_train', 'lento', 'variante_v2']
])


,pista,id,detector,familia,n_features,step,ventana_train,lento,variante_v2
0,A,D01,Regla de volatilidad/VIX,Reglas,1,21,expanding,False,A: umbral de volatilidad realizada; VIX no existe en la pista histórica
1,A,D02,Regla compuesta risk-off,Reglas,4,21,expanding,False,
2,A,D03,GMM (K=3),Clustering,9,21,expanding,False,
3,A,D04,HMM gaussiano (K=2),HMM,9,21,expanding,False,
4,A,D05,Markov-Switching varianza,Markov-Switching,1,63,expanding,True,
5,A,D06,GJR-GARCH-t,GARCH,1,21,expanding,False,
6,A,D07,CUSUM online robusto,Change-point,1,21,expanding,False,
7,A,D08,HMM t-Student (K=4),HMM,9,126,expanding,True,
8,A,D09,Statistical Jump Model,Jump model,9,21,expanding,False,
9,A,D10,Turbulencia Mahalanobis,Reglas multivariantes,4,21,expanding,False,


## 2. Cómo se implementa la comparación

Para cada pista y detector se aplica el mismo esquema:

1. se unen los bloques diario y mensual ya alineados por `03_preprocesado`;
2. se seleccionan únicamente las columnas declaradas para ese detector;
3. se fija una única rejilla completa usando la unión de features de los doce, para que todos compartan exactamente las mismas fechas OOS, sin imputar valores;
4. cada fold ajusta una instancia nueva usando exclusivamente observaciones
   anteriores al bloque OOS;
5. `predict_online` produce etiquetas diarias sin suavizado futuro;
6. las crisis y trampas se cargan desde `benchmark_spec.yaml` para la pista activa;
7. se guarda inmediatamente el resultado antes de continuar con el siguiente modelo.

La ventana inicial es común dentro de cada pista: ocho años en A y tres en B. La
ventana corta de B implica que la GFC queda en entrenamiento, no OOS; se declarará
como `NaN` y no se premiará ni penalizará. Los valores son configuración de benchmark,
no hiperparámetros elegidos mirando las métricas.

Consecuencias que conviene tener presentes al leer el notebook 05:

- **Crisis realmente evaluadas.** El OOS de A empieza el 1970-05-12: `credit_crunch_1966`
  queda fuera y `bear_1969_70` solo aporta sus 11 últimos días (de 370), así que A
  puntúa 17 episodios, uno de ellos parcial. El OOS de B empieza el 2010-04-12: se
  evalúan 9 de sus 10 crisis (sin la GFC).
- **Frecuencia de refit.** D5 reestima cada 63 sesiones y D8/D11 cada 126 (D11 además con
  ventana móvil). No cambia las fechas OOS, pero sí `label_stability`, que compara la
  etiqueta de un bloque con la que le da el modelo siguiente: con refits más espaciados
  el modelo siguiente ha visto más datos nuevos.
- **Estado de los autómatas en cada refit.** Las reglas con histéresis (D1, D2, D6, D10)
  reinician su autómata en calma al comenzar cada bloque OOS; D7 sí propaga su CUSUM
  desde el train. Es causal, pero penaliza a esas reglas (ver limitaciones en 05).


In [3]:
# Interruptores deliberados. Con CACHE_ONLY=True, abrir o ejecutar todas las celdas NO
# lanza horas de cálculo ni sobrescribe results/: solo se leen las cachés cuya huella
# (código + datos + entorno) se verifica en esta máquina. Para recalcular (~4 h) hay
# que poner CACHE_ONLY=False de forma explícita.
RUN_BENCHMARK = True
CACHE_ONLY = True             # True = solo lectura; False = ajusta lo que falte
FORCE = False                 # False reutiliza cada CSV ya terminado
N_JOBS = 1                    # >1 = un proceso por detector (ver §4); solo si CACHE_ONLY=False
TRACKS = ('A', 'B')
DETECTOR_IDS = tuple(f'D{i:02d}' for i in range(1, 13))
TRAIN_DAYS = {'A': 252 * 8, 'B': 252 * 3}
OUTPUT_DIR = ROOT / 'results' / 'benchmark_v2'

print('RUN_BENCHMARK =', RUN_BENCHMARK, '| CACHE_ONLY =', CACHE_ONLY, '| N_JOBS =', N_JOBS)
print('Detectores:', DETECTOR_IDS)
print('Train inicial:', TRAIN_DAYS)
print('Salida:', OUTPUT_DIR.relative_to(ROOT))


RUN_BENCHMARK = True
Detectores: ('D01', 'D02', 'D03', 'D04', 'D05', 'D06', 'D07', 'D08', 'D09', 'D10', 'D11', 'D12')
Train inicial: {'A': 2016, 'B': 756}
Salida: results\benchmark_v2


## 3. Gate previo: datos, columnas y dependencias

El benchmark no debe empezar parcialmente sin avisar. Este gate comprueba que los
cuatro Parquet de `data/processed/` existen, que cada configuración encuentra sus
features y que están instaladas las dependencias opcionales. En particular, D6
requiere `arch` y D9 requiere `jumpmodels`.

Si faltan los paneles —situación normal después de clonar porque están ignorados por
Git— hay que ejecutar primero `python -m src.ingest.download` y después `03_preprocesado.ipynb`.


In [4]:
preflight = pd.concat([bm.preflight(track) for track in TRACKS], ignore_index=True)
display(preflight)
HAY_DATOS = bool(preflight['datos_ok'].all())
GATE_OK = bool(HAY_DATOS and preflight['features_ok'].all() and preflight['dependencia_ok'].all())
print('Gate completo:', 'OK' if GATE_OK else 'PENDIENTE')
if not HAY_DATOS:
    print('Acción: ejecutar python -m src.ingest.download y después notebooks/03_preprocesado.ipynb.')
if HAY_DATOS and not preflight['dependencia_ok'].all():
    print('Acción: pip install -r requirements.txt')


,pista,id,datos_ok,features_ok,dependencia_ok,detalle
0,A,D01,True,True,True,OK
1,A,D02,True,True,True,OK
2,A,D03,True,True,True,OK
3,A,D04,True,True,True,OK
4,A,D05,True,True,True,OK
5,A,D06,True,True,True,OK
6,A,D07,True,True,True,OK
7,A,D08,True,True,True,OK
8,A,D09,True,True,True,OK
9,A,D10,True,True,True,OK


Gate completo: OK


## 4. Ejecución reanudable

La celda siguiente recorre la matriz pista × detector. `run_status.csv` se actualiza
tras cada intento y cada CSV individual actúa como *checkpoint*. Si una ejecución se
interrumpe, se vuelve a lanzar con `FORCE=False` y continúa desde el primer resultado
ausente. Los errores se registran y no borran resultados previos. Con `CACHE_ONLY=True` la celda
es de solo lectura: no ajusta modelos ni reescribe `manifest.json` ni `run_status.csv`.

Para una prueba corta puede cambiarse temporalmente, por ejemplo,
`DETECTOR_IDS = ('D01', 'D07')`. La ejecución oficial debe restaurar los doce.


### Re-ejecución completa en paralelo (recomendada tras ADR-003)

La huella de caché cambió en ADR-003 (esquema 3: contenido de los parquet en vez de
bytes, `detector_base` realmente importado, columnas `det_*` de detección por evento),
así que **todas** las cachés versionadas quedan invalidadas y hay que recalcular una vez.
Lo más cómodo es hacerlo fuera del notebook, desde la raíz del repo:

```bash
# matriz completa, 4 procesos (lentos primero); manifest.json y run_status.csv los
# escribe solo el proceso principal, cada hijo escribe su CSV, su panel y status/<P>_<D>.json
python -m src.benchmark --track A B --jobs 4

# alternativa: un detector por terminal (no toca manifest/run_status) y consolidar al final
python -m src.benchmark --track A --detector D04
python -m src.benchmark --track A --detector D09
python -m src.benchmark --consolidate --track A B
```

`--jobs` limita cada hijo a 1 hilo BLAS/OpenMP (`--threads-per-job` para cambiarlo) y
`--force` ignora la caché. Después basta con volver a abrir este notebook con
`CACHE_ONLY=True` (leerá las cachés nuevas) y ejecutar `05`. Desde el notebook equivale a
`CACHE_ONLY=False` y `N_JOBS=4`.

In [5]:
if RUN_BENCHMARK:
    if not GATE_OK:
        raise RuntimeError('El gate previo no está en verde; corrige datos/dependencias antes de ejecutar.')
    metrics_run, run_status = bm.run_benchmark(
        tracks=TRACKS,
        detector_ids=DETECTOR_IDS,
        output_dir=OUTPUT_DIR,
        train_days=TRAIN_DAYS,
        force=FORCE,
        continue_on_error=True,
        cache_only=CACHE_ONLY,
        n_jobs=N_JOBS,
    )
    if CACHE_ONLY and (run_status['estado'] != 'cache').any():
        print('Modo solo-caché: hay combinaciones sin caché verificable en esta máquina '
              '(huella distinta o paneles OOS ausentes). No se ha recalculado nada; '
              '05 usará las métricas versionadas y lo indicará.')
    display(run_status)
else:
    print('Ejecución desactivada. Cambia RUN_BENCHMARK=True cuando el gate esté en verde.')


,pista,id,estado,segundos,detalle
0,A,D01,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D01.csv
1,A,D02,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D02.csv
2,A,D03,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D03.csv
3,A,D04,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D04.csv
4,A,D05,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D05.csv
5,A,D06,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D06.csv
6,A,D07,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D07.csv
7,A,D08,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D08.csv
8,A,D09,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D09.csv
9,A,D10,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D10.csv


## 5. Inventario de resultados disponibles

Esta sección puede ejecutarse aunque el benchmark esté incompleto. Permite ver qué
detectores terminaron, cuáles se recuperaron de caché validada por huella y qué errores necesitan una
repetición. No construye rankings parciales silenciosamente: la comparación del
notebook 05 mostrará expresamente cuántas filas hay por pista.


In [6]:
status_path = OUTPUT_DIR / 'run_status.csv'
if status_path.exists():
    status = pd.read_csv(status_path)
    display(status)
    print('\nConteo por estado:')
    display(status.groupby(['pista', 'estado']).size().rename('n').reset_index())
else:
    print('Todavía no existe run_status.csv.')

metric_files = sorted((OUTPUT_DIR / 'metrics').glob('[AB]_D*.csv'))
print(f'CSV de métricas disponibles: {len(metric_files)}/24')
for path in metric_files:
    print(' -', path.relative_to(ROOT))


,pista,id,estado,segundos,detalle
0,A,D01,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D01.csv
1,A,D02,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D02.csv
2,A,D03,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D03.csv
3,A,D04,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D04.csv
4,A,D05,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D05.csv
5,A,D06,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D06.csv
6,A,D07,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D07.csv
7,A,D08,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D08.csv
8,A,D09,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D09.csv
9,A,D10,cache,0.0,caché verificada: results\benchmark_v2\metrics\A_D10.csv



Conteo por estado:


,pista,estado,n
0,A,cache,12
1,B,cache,12


CSV de métricas disponibles: 24/24
 - results\benchmark_v2\metrics\A_D01.csv
 - results\benchmark_v2\metrics\A_D02.csv
 - results\benchmark_v2\metrics\A_D03.csv
 - results\benchmark_v2\metrics\A_D04.csv
 - results\benchmark_v2\metrics\A_D05.csv
 - results\benchmark_v2\metrics\A_D06.csv
 - results\benchmark_v2\metrics\A_D07.csv
 - results\benchmark_v2\metrics\A_D08.csv
 - results\benchmark_v2\metrics\A_D09.csv
 - results\benchmark_v2\metrics\A_D10.csv
 - results\benchmark_v2\metrics\A_D11.csv
 - results\benchmark_v2\metrics\A_D12.csv
 - results\benchmark_v2\metrics\B_D01.csv
 - results\benchmark_v2\metrics\B_D02.csv
 - results\benchmark_v2\metrics\B_D03.csv
 - results\benchmark_v2\metrics\B_D04.csv
 - results\benchmark_v2\metrics\B_D05.csv
 - results\benchmark_v2\metrics\B_D06.csv
 - results\benchmark_v2\metrics\B_D07.csv
 - results\benchmark_v2\metrics\B_D08.csv
 - results\benchmark_v2\metrics\B_D09.csv
 - results\benchmark_v2\metrics\B_D10.csv
 - results\benchmark_v2\metrics\B_D11.csv

## 6. Cierre

Este notebook termina cuando existen 24 filas válidas —12 detectores por dos pistas—
o cuando cualquier excepción queda documentada como incompatibilidad explícita. No
se interpreta aquí el ganador: `05_comparacion_detectores.ipynb` carga los artefactos,
verifica comparabilidad dentro de cada pista y construye la evidencia visual.

La prueba pseudolive estricta del detector finalmente elegido será posterior a esta
comparación y utilizará un tramo final congelado que no intervenga en la selección.
